# DS4DS Exercise Sheet 6


**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

In [ ]:
using OrdinaryDiffEq
using MAT
using LinearAlgebra

### Task 1: OLS for the nonlinear pendulum

We once again consider the nonlinear, frictionless pendulum that we have already seen in Exercise 02. The state-space representation of the pendulum is given by

$$
\begin{align}
    \mathbf{x}(t) = \begin{bmatrix}\theta(t)\\ \frac{\mathrm{d}\theta(t)}{\mathrm{d}t} \end{bmatrix}
\end{align}
$$

and

$$
\begin{align}
    \frac{\mathrm{d}\mathbf{x}(t)}{\mathrm{d}t} = \begin{bmatrix}x_2(t) \\ -\frac{g}{L} \sin{(x_1(t))} \end{bmatrix}
\end{align}
$$

where $g = 9.81 \frac{\mathrm{m}}{\mathrm{s}^2}$ is the magnitude of the gravitational field, $L = 1 \mathrm{m}$ is the length of the rod, $\theta$ is the angle from the vertical axis, and $\frac{\mathrm{d}\mathbf{\theta}(t)}{\mathrm{d}t}$ is the angular velocity.

In [ ]:
# Model parameters defined as constants, i.e., they can be also used within function calls
const g = 9.81; # gravity constant
const L = 1.0; # length of pendulum

**a)** Now we want to study the system using data. To this end, implement the function `Tsit5_ode_solver` and the function `pendulum!` (which contains the right-hand side of the ODE) outlined below. Within your solution, you are to use the `Tsit5()`-ODE Solver from the `OrdinaryDiffEq`-package. The state trajectory must be specified at the times $t=0, \Delta t, 2 \cdot \Delta t \dots$.

**Hint:** Here you can find out how to use the package: https://docs.sciml.ai/DiffEqDocs/stable/.

In [ ]:
# Define the ODE model shown above as a function which can be passed to a solver

function pendulum!(dx, x, p, t)
    """Consider the examples at https://github.com/SciML/OrdinaryDiffEq.jl or examples from the lecture for guidance."""

    ### BEGIN SOLUTION

    dx[1] = x[2]
    dx[2] = -(g/L)*sin(x[1])
    nothing

    ### END SOLUTION

end


function Tsit5_ode_solver(ode_function, x0, tspan, dt)
    """Defines the ODE problem and solves it.
    
    Args:
        ode_function: The ODE defined as a function that can be passed to the solver
        x0: Initial state of the system
        tspan: time-span in which the system should be evaluated (This is a tuple with two values (t_0, t_e))
        dt: The length of each of the timesteps
    
    Returns:
        t: A vector containing the time for the corresponding elements in the state trajectory
        x: State trajectory as a Matrix in the form (n_states, N) where N is the length of the trajectory
    """

    ### BEGIN SOLUTION

    prob = ODEProblem(ode_function, x0, tspan);
    sol = solve(prob, Tsit5(), saveat=dt);

    t, x = sol.t, sol.u
    x = reduce(hcat,x)

    ### END SOLUTION

    return t, x
end

**b)** Use the functions implemented above to simulate the system on the time horizon $[t_0, t_e] = [0 \mathrm{s}, 2.5 \mathrm{s}]$ with $\Delta t = 0.01 \mathrm{s}$ and create trajectory data for different initial conditions for $\mathbf{x}_0$

- i.)  $ \,\, \mathbf{x}_{0,i} = \begin{bmatrix}\frac{\pi}{9}\\ 0 \end{bmatrix}$
- ii.) $ \, \mathbf{x}_{0,ii} = \begin{bmatrix}\frac{\pi}{4}\\ 0 \end{bmatrix}$
- iii.) $ \mathbf{x}_{0,iii} = \begin{bmatrix}\frac{\pi}{2}\\ 0 \end{bmatrix}$.

Find the period $T$ with which the system oscillates for the different initial conditions. To this end, find the first point $t$ when $\theta(t) \approx \theta_0$ (take the point from your discrete trajectory that is closest to this value with regard to the 2-norm of the error). Save your results for $T$ in the variables $T_i$, $T_{ii}$ and $T_{iii}$ respectively.

In [ ]:
function get_T_estimate(x0)

    ### BEGIN SOLUTION

    t, x = Tsit5_ode_solver(pendulum!, x0, tspan, dt)
    difference = x .- x0

    norm_per_sample = zeros(size(difference, 2))
    for i in 1:size(difference, 2)
        norm_per_sample[i] = sum(difference[:, i].^2)
    end

    min_idx = argmin(norm_per_sample[2:end])
    T = t[min_idx]

    ### END SOLUTION

    return T, x
end

### BEGIN SOLUTION

# Time settings
tspan = (0.0, 2.5)
dt = 0.01;

x0_i = [π/9, 0]
x0_ii = [π/4, 0]
x0_iii = [π/2, 0]

T_i, x_i = get_T_estimate(x0_i);
T_ii, x_ii = get_T_estimate(x0_ii);
T_iii, x_iii = get_T_estimate(x0_iii);

### END SOLUTION

**c)** Use the OLS approach to identify the matrix $\mathbf{\tilde{A}}$ of an approximated linear system of the form $\mathbf{x}[k+1] = \mathbf{\tilde{A}} \mathbf{x}[k]$. Use your trajectories from a) and identify one matrix per trajectory.

In [ ]:
function estimate_A_tilde_OLS(x)

    ### BEGIN SOLUTION

    A_tilde = x[:, 2:end] * pinv(x[:, 1:end-1]);

    ### END SOLUTION

    return A_tilde
end

### BEGIN SOLUTION

A_tilde_i = estimate_A_tilde_OLS(x_i);
A_tilde_ii = estimate_A_tilde_OLS(x_ii);
A_tilde_iii = estimate_A_tilde_OLS(x_iii);

### END SOLUTION

In [ ]:
x_test = [-1,2] .* [1, 2, -3, 4]'
A_tilde_test = estimate_A_tilde_OLS(x_test)

**d)** Approximate the eigenfrequencies $\omega$ of the system for the three initial conditions given above using the $\mathbf{\tilde{A}}$ matrices. Then use these frequencies to approximate the oscillating periods $\tilde{T}$ of the learned linear systems.

In [ ]:
function calculate_frequency(A_tilde)

    ### BEGIN SOLUTION

    mu, vecs = eigen(A_tilde)
    
    lambda = log.(mu) / dt
    omega = imag(lambda)
    
    # Alternativ solution
    # angles = angle.(eigenvalues) 
    # omega = maximum(abs.(angles)) / 0.01

    @assert abs(omega[1]) == abs(omega[2])
    omega = abs(omega[1])

    ### END SOLUTION

    return omega
end

### BEGIN SOLUTION

omega_i = calculate_frequency(A_tilde_i);
omega_ii = calculate_frequency(A_tilde_ii);
omega_iii = calculate_frequency(A_tilde_iii);

T_tilde_i = 2 * π / omega_i
T_tilde_ii = 2 * π / omega_ii
T_tilde_iii = 2 * π / omega_iii

### END SOLUTION

**e)** Determine the relative error between the estimates for the oscillating periods from subtask b) and d). Consider the value from b) to be the truth.

In [ ]:
### BEGIN SOLUTION

REs = []
for (T_tilde, T) in zip([T_tilde_i, T_tilde_ii, T_tilde_iii], [T_i, T_ii, T_iii])
    rel_error = abs(T_tilde - T) / T
    append!(REs, rel_error)
end

rel_error_i, rel_error_ii, rel_error_iii = REs;

### END SOLUTION

**f)** Use the estimated matrices from c) to predict the system behavior for the three different initial conditions over the same timespan as before. Compute the MSE between the true trajectory and the linear approximation.

The MSE is defined as

$$
\begin{align}
    \mathrm{MSE}(\begin{bmatrix}\mathbf{x}[1] & \dots & \mathbf{x}[N]\end{bmatrix}, \begin{bmatrix}\mathbf{\tilde{x}}[1] & \dots & \mathbf{\tilde{x}}[N]\end{bmatrix}) 
    = \frac{1}{N} \sum_{k=1}^N \| \mathbf{x}[k] - \mathbf{\tilde{x}}[k] \|_2^2
\end{align}.
$$

In [ ]:

### BEGIN SOLUTION

function get_approximated_trajectory(A_tilde, x0, tspan, dt)
    n_states = length(x0)
    N = Int((tspan[2] - tspan[1]) / dt + 1)
    x_approx = zeros((n_states, N))
    x_approx[:, 1] = x0
    for j in 2:N
        x_approx[:, j] = A_tilde * x_approx[:, j-1]
#         println(isapprox(x_approx[:, j], A_tilde^(j-1) * x0))
    end
    
    return x_approx
end

x_approx_i = get_approximated_trajectory(A_tilde_i, x0_i, tspan, dt)
x_approx_ii = get_approximated_trajectory(A_tilde_ii, x0_ii, tspan, dt)
x_approx_iii = get_approximated_trajectory(A_tilde_iii, x0_iii, tspan, dt)


function MSE(x, y)
    return sum((x-y).^2) / size(x, 2)
end

trajectory_MSE_i = MSE(x_approx_i, x_i);
trajectory_MSE_ii = MSE(x_approx_ii, x_ii);
trajectory_MSE_iii = MSE(x_approx_iii, x_iii);

### END SOLUTION

## 2) DMD of the flow past a cylinder

Perform a DMD analysis on the vortex shedding data set. This phenomenon occurs frequently in nature, when a fluid hits a bluff body (see, e.g., the following NASA video on cloud patterns: https://www.youtube.com/watch?v=SawKLWT1bDA). The data set consists of 50 snapshots (with $\Delta t = 0.1$) of the absolute velocity of a fluid entering from the left and flowing around a cylinder. Each snapshot contains velocity measurements on a $449 x 199$ grid. In vectorized form, this leads to a data matrix $Z \in \mathbb{R}^{89351 \times 151}$.

In [ ]:
# load the data
data = matread("VortexShedding.mat")
data = data["Z"];

You may use this code to produce an example plot of the initial state:
```
display(heatmap(reshape(data[:, 1], 199, 449), aspect_ratio=1, color=:auto, title="Initial state"))
```
<img src="./initial_state.png" alt="initial_state" width="800"/>

Additionally, if you want to take a closer look at the data, there is an animation of the data as a gif in the exercise folder.

### Task a) 
Decompose the data matrix into the two time-shifted versions $Z$ and $Z’$ and compute a singular value decomposition of $Z$. Store the 10 largest singular values $\sigma$ in a separate array.

**Hint: Use the parameters created here in the course of further tasks instead of executing SVD again.**

In [ ]:
### BEGIN SOLUTION
Z = data[:, 1:end-1]
Z_shift = data[:, 2:end]
U, S, V = svd(Z)
largest10singularValues = S[1:10]
### END SOLUTION

### Task b)
How many singular vectors are at least required to reconstruct 1) 90% and 2) 99% of the original information? (_Hint: Take a look at the singular values_)

In [ ]:
### BEGIN SOLUTION
print(sum(S[1:2])/sum(S)," ") 
println(sum(S[1:3])/sum(S))
r_90 = 3

print(sum(S[1:10])/sum(S)," ") 
println(sum(S[1:11])/sum(S)) 
r_99 = 11
### END SOLUTION

### Task c)
Identify a linear model with system matrix $\tilde{A}$ from the data via dynamic mode decomposition. Set the rank of the reduced DMD method to $r=20$. Store your sulution in `tilde_A`

In [ ]:
### BEGIN SOLUTION
r = 20
Ur = U[:, 1:r]
Sr = S[1:r]
Vr = V[:, 1:r]
tilde_A = Ur' * Z_shift * Vr * diagm(1 ./ Sr)
### END SOLUTION

### Task d)
Sort the eigenvalues of $\tilde{A}$ according to their frequency in ascending order (starting with 0). Which is the lowest frequency $\omega$ in the system dynamics (aside from the stationary mode with $\lambda = 0$)? In addition, calculate the corresponding period $T$ with which the system oscillates.

In [ ]:
### BEGIN SOLUTION
μ, P = eigen(tilde_A)
λ = log.(μ) / 0.1
ω = imag.(λ)
i_sort = sortperm(ω, by=abs)
lowest_freq = ω[end-1]
T =  2 * π / lowest_freq
### END SOLUTION

### Task e)
Use the reduced model for prediction via the “project-map-lift” approach. To this end:
1. Project the initial state onto the $r$-dimensional subspace,
2. Simulate the low-dimensional system for 20 time steps,
3. Lift the final state last state to the original space,
4. Calculate the error (using the 2-norm) between the the reduced order model solution and the original data at that time step.

In [ ]:

### BEGIN SOLUTION
simulated_values = zeros(20,21)
#i
simulated_values[:,1] = Ur' * data[:, 1]
#ii
for i in 2:21
    simulated_values[:,i] = tilde_A * simulated_values[:,i-1]
end
#iii
Z_future = Ur * simulated_values[:,21]
#iv
error = norm(data[:, 21] - Z_future, 2);

# Alternativ solution:
# projected_x0 = Ur' * data[:, 1]
# mapped_x20 = tilde_A^20 * projected_x0
# lifted_x20 = Ur * mapped_x20
# original_x20 = data[:, 21]

# Typical error:
# Don't consider the 20 steps -> using data[:,20] = 19 steps

### END SOLUTION